In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.sql("""
SELECT a.*
FROM policyprojcatalog.policyprojdb.agent a
INNER JOIN policyprojcatalog.policyprojdb.branch b
    ON a.branch_id = b.branch_id
WHERE a.merge_flag = false
  AND length(a.agent_phone) = 10
""")

display(df)

In [0]:
df.createOrReplaceTempView("agent_temp")

df_email = spark.sql("""
SELECT
    agent_id,
    agent_name,
    agent_phone,
    branch_id,
    create_timestamp,
    CASE 
        WHEN agent_email = '' OR agent_email IS NULL 
        THEN 'Shrey@gmail.com'
        ELSE agent_email
    END AS agent_email
FROM agent_temp
""")

display(df_email)

In [0]:
df_email.createOrReplaceTempView("clean_agent")

spark.sql("""
MERGE INTO policyprojcatalog.silver.agent AS T
USING clean_agent AS S
ON T.agent_id = S.agent_id

WHEN MATCHED THEN UPDATE SET
    T.agent_phone = S.agent_phone,
    T.agent_email = S.agent_email,
    T.agent_name = S.agent_name,
    T.branch_id = S.branch_id,
    T.create_timestamp = S.create_timestamp,
    T.merged_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    agent_id,
    agent_name,
    agent_email,
    agent_phone,
    branch_id,
    create_timestamp,
    merged_timestamp
)
VALUES (
    S.agent_id,
    S.agent_name,
    S.agent_email,
    S.agent_phone,
    S.branch_id,
    S.create_timestamp,
    current_timestamp()
)
""")

In [0]:
spark.sql("""
UPDATE policyprojcatalog.policyprojdb.agent
SET merge_flag = true
WHERE merge_flag = false
""")